In [0]:
CREATE OR REPLACE TABLE workspace.dataforge.clean_insurance_data AS
SELECT 
TRIM(Customer_ID) AS Customer_ID,

CAST(TRIM(Age) AS INT) AS Age,

TRIM(coalesce(CASE WHEN TRIM(Gender) like 'M%' THEN 'Male'
WHEN TRIM(Gender) like 'F%' THEN 'Female'
ELSE TRIM(Gender)
END,
(SELECT MODE(TRIM(Gender)) FROM workspace.dataforge.dfa_insurance_project_raw_dataset))) AS Gender,

TRIM(coalesce(CASE
WHEN Trim(UPPER(Province)) IN ("GP", "GAUTENG") THEN "Gauteng"
WHEN Trim(UPPER(Province)) IN ("EC", "EASTERN CAPE") THEN "Eastern Cape"
WHEN Trim(UPPER(Province)) IN ("FS", "FREE STATE") THEN "Free State"
WHEN Trim(UPPER(Province)) IN ("NW", "NORTH WEST") THEN "North West"
WHEN Trim(UPPER(Province)) IN ("WC", "WESTERN CAPE") THEN "Western Cape"
WHEN Trim(UPPER(Province)) IN ("MP", "MPUMALANGA") THEN "Mpumalanga"
WHEN Trim(UPPER(Province)) IN ("LP", "LIMPOPO") THEN "Limpopo"
WHEN Trim(UPPER(Province)) IN ("KZN", "KWAZULU-NATAL", "KWAZULU NATAL") THEN "KwaZulu-Natal"
END,
(SELECT mode(trim(CASE
WHEN Trim(UPPER(Province)) IN ("GP", "GAUTENG") THEN "Gauteng"
WHEN Trim(UPPER(Province)) IN ("EC", "EASTERN CAPE") THEN "Eastern Cape"
WHEN Trim(UPPER(Province)) IN ("FS", "FREE STATE") THEN "Free State"
WHEN Trim(UPPER(Province)) IN ("NW", "NORTH WEST") THEN "North West"
WHEN Trim(UPPER(Province)) IN ("WC", "WESTERN CAPE") THEN "Western Cape"
WHEN Trim(UPPER(Province)) IN ("MP", "MPUMALANGA") THEN "Mpumalanga"
WHEN Trim(UPPER(Province)) IN ("LP", "LIMPOPO") THEN "Limpopo"
WHEN Trim(UPPER(Province)) IN ("KZN", "KWAZULU-NATAL", "KWAZULU NATAL") THEN "KwaZulu-Natal"
END)) FROM workspace.dataforge.dfa_insurance_project_raw_dataset))) AS Province,

CAST(TRIM(coalesce(CASE 
  WHEN Monthly_Income LIKE '%R%' THEN CAST(REGEXP_REPLACE(Monthly_Income, '[R, ]', '') AS DOUBLE)
  WHEN Monthly_Income LIKE '%,%' THEN CAST(REGEXP_REPLACE(Monthly_Income, ',', '') AS DOUBLE)
  ELSE CAST(Monthly_Income AS DOUBLE)
END,
(SELECT mean(Trim(CASE 
  WHEN Monthly_Income LIKE '%R%' THEN CAST(REGEXP_REPLACE(Monthly_Income, '[R, ]', '') AS DOUBLE)
  WHEN Monthly_Income LIKE '%,%' THEN CAST(REGEXP_REPLACE(Monthly_Income, ',', '') AS DOUBLE)
  ELSE CAST(Monthly_Income AS DOUBLE)
END)) FROM workspace.dataforge.dfa_insurance_project_raw_dataset))) AS DOUBLE) AS Monthly_Income,

CASE
WHEN substring(trim(Join_Date),3,1) = "/" THEN to_date(trim(Join_Date), 'dd/MM/yyyy')
WHEN substring(trim(Join_Date),5,1) = "-" THEN to_date(trim(Join_Date), 'yyyy-MM-dd')
WHEN substring(trim(Join_Date),5,1) = "/" THEN to_date(trim(Join_Date), 'yyyy/MM/dd')
WHEN trim(Join_Date) LIKE '__-___-____' THEN to_date(trim(Join_Date), 'dd-MMM-yyyy')
WHEN substring(trim(Join_Date),3,1) = "-" THEN to_date(trim(Join_Date), 'MM-dd-yyyy')
END AS Join_Date,

TRIM(Policy_ID) AS Policy_ID,

TRIM(initcap(Policy_Type)) AS Policy_Type,

CAST(TRIM(coalesce(CASE 
  WHEN Premium_Amount LIKE '%R%' THEN CAST(REGEXP_REPLACE(Premium_Amount, '[R, ]', '') AS DOUBLE)
  WHEN Premium_Amount LIKE '%,%' THEN CAST(REGEXP_REPLACE(Premium_Amount, ',', '') AS DOUBLE)
  ELSE CAST(Premium_Amount AS DOUBLE)
END,
(SELECT mean(Trim(CASE 
  WHEN Premium_Amount LIKE '%R%' THEN CAST(REGEXP_REPLACE(Premium_Amount, '[R, ]', '') AS DOUBLE)
  WHEN Premium_Amount LIKE '%,%' THEN CAST(REGEXP_REPLACE(Premium_Amount, ',', '') AS DOUBLE)
  ELSE CAST(Premium_Amount AS DOUBLE)
END)) FROM workspace.dataforge.dfa_insurance_project_raw_dataset))) AS DOUBLE) AS Premium_Amount,

TRIM(CASE WHEN Trim(initcap(Policy_Status)) = "Cancelled" THEN "Canceled" 
Else Trim(initcap(Policy_Status))
END) AS Policy_Status,

TRIM(initcap(Claim_ID)) AS Claim_ID,

CASE
WHEN substring(trim(Claim_Date),3,1) = "/" THEN to_date(trim(Claim_Date), 'dd/MM/yyyy')
WHEN substring(trim(Claim_Date),5,1) = "-" THEN to_date(trim(Claim_Date), 'yyyy-MM-dd')
WHEN substring(trim(Claim_Date),5,1) = "/" THEN to_date(trim(Claim_Date), 'yyyy/MM/dd')
WHEN trim(Claim_Date) LIKE '__-___-____' THEN to_date(trim(Claim_Date), 'dd-MMM-yyyy')
WHEN substring(trim(Claim_Date),3,1) = "-" THEN to_date(trim(Claim_Date), 'MM-dd-yyyy')
END AS Claim_Date,

CAST(TRIM(coalesce(CASE 
  WHEN Claim_Amount LIKE '%R%' THEN CAST(REGEXP_REPLACE(Claim_Amount, '[R, ]', '') AS DOUBLE)
  WHEN Claim_Amount LIKE '%,%' THEN CAST(REGEXP_REPLACE(Claim_Amount, ',', '') AS DOUBLE)
  WHEN Claim_Amount LIKE '%-%' THEN CAST(REGEXP_REPLACE(Claim_Amount, '-', '') AS DOUBLE)
  ELSE CAST(Claim_Amount AS DOUBLE)
END,
(SELECT mean(Trim(CASE 
  WHEN Claim_Amount LIKE '%R%' THEN CAST(REGEXP_REPLACE(Claim_Amount, '[R, ]', '') AS DOUBLE)
  WHEN Claim_Amount LIKE '%,%' THEN CAST(REGEXP_REPLACE(Claim_Amount, ',', '') AS DOUBLE)
  WHEN Claim_Amount LIKE '%-%' THEN CAST(REGEXP_REPLACE(Claim_Amount, '-', '') AS DOUBLE)
  ELSE CAST(Claim_Amount AS DOUBLE)
END)) FROM workspace.dataforge.dfa_insurance_project_raw_dataset))) AS DOUBLE) AS Claim_Amount,

TRIM(CASE WHEN Trim(initcap(Claim_Status)) = "In Review" THEN "Pending"
WHEN Trim(initcap(Claim_Status)) = "Declined" THEN "Rejected"
ELSE Trim(initcap(Claim_Status))
END) AS Claim_Status,

TRIM(coalesce(CASE WHEN TRIM(Fraud_Flag) like 0 THEN 'No'
WHEN TRIM(Fraud_Flag) like 1 THEN "Yes"
WHEN TRIM(Fraud_Flag) like "Y%%" THEN "Yes"
WHEN TRIM(Fraud_Flag) like "N%" THEN "No"
WHEN TRIM(Fraud_Flag) like "N" THEN "No"
WHEN TRIM(Fraud_Flag) like "no" THEN "No"
ELSE TRIM(Fraud_Flag)
END,
(SELECT MODE(TRIM(Fraud_Flag)) FROM workspace.dataforge.dfa_insurance_project_raw_dataset))) AS Fraud_Flag
FROM workspace.dataforge.dfa_insurance_project_raw_dataset;

SELECT * FROM workspace.dataforge.clean_insurance_data

num_affected_rows,num_inserted_rows


Customer_ID,Age,Gender,Province,Monthly_Income,Join_Date,Policy_ID,Policy_Type,Premium_Amount,Policy_Status,Claim_ID,Claim_Date,Claim_Amount,Claim_Status,Fraud_Flag
CUST1000,44,Male,Gauteng,26064.0,2020-05-30,POL5000,Home,1495.0,Active,Clm9000,2024-04-14,20438.99,Approved,No
CUST1001,41,Male,Free State,3500.0,2025-06-16,POL5001,Auto,746.28,Active,Clm9001,2025-09-08,4583.11,Approved,No
CUST1002,33,Female,Limpopo,29552.92,2020-06-14,POL5002,Health,2325.42,Active,Clm9002,2024-06-12,11892.02,Rejected,No
CUST1003,41,Female,Western Cape,3500.0,2024-06-10,POL5003,Auto,885.13,Active,Clm9003,2025-01-24,15139.51,Approved,No
CUST1004,42,Female,Mpumalanga,3500.0,2024-07-15,POL5004,Home,1875.25,Active,Clm9004,2025-01-22,29975.0,Approved,No
CUST1005,33,Female,Limpopo,27732.53939962478,2024-04-13,POL5005,Home,2210.31,Active,Clm9005,2024-10-23,30336.75,Approved,No
CUST1006,38,Male,Free State,49905.01,2022-12-19,POL5006,Motor,433.09,Active,Clm9006,2025-01-14,500.0,Approved,No
CUST1007,30,Female,KwaZulu-Natal,20975.4,2021-10-14,POL5007,Life,817.56,Active,Clm9007,2024-06-16,39102.08,Rejected,No
CUST1008,42,Male,North West,31654.77,2022-03-28,POL5008,Home,1410.71,Active,Clm9008,2024-02-02,48634.23,Approved,No
CUST1009,41,Male,Limpopo,26957.76,2020-11-02,POL5009,Funeral,326.92,Active,Clm9009,2024-08-05,22870.665962962965,Approved,No


In [0]:
DELETE FROM workspace.dataforge.clean_insurance_data WHERE Customer_ID = '';
DELETE FROM workspace.dataforge.clean_insurance_data WHERE Age < 18;
SELECT * FROM workspace.dataforge.clean_insurance_data

num_affected_rows
3


num_affected_rows
1


Customer_ID,Age,Gender,Province,Monthly_Income,Join_Date,Policy_ID,Policy_Type,Premium_Amount,Policy_Status,Claim_ID,Claim_Date,Claim_Amount,Claim_Status,Fraud_Flag
CUST1000,44,Male,Gauteng,26064.0,2020-05-30,POL5000,Home,1495.0,Active,Clm9000,2024-04-14,20438.99,Approved,No
CUST1001,41,Male,Free State,3500.0,2025-06-16,POL5001,Auto,746.28,Active,Clm9001,2025-09-08,4583.11,Approved,No
CUST1002,33,Female,Limpopo,29552.92,2020-06-14,POL5002,Health,2325.42,Active,Clm9002,2024-06-12,11892.02,Rejected,No
CUST1003,41,Female,Western Cape,3500.0,2024-06-10,POL5003,Auto,885.13,Active,Clm9003,2025-01-24,15139.51,Approved,No
CUST1004,42,Female,Mpumalanga,3500.0,2024-07-15,POL5004,Home,1875.25,Active,Clm9004,2025-01-22,29975.0,Approved,No
CUST1005,33,Female,Limpopo,27732.53939962478,2024-04-13,POL5005,Home,2210.31,Active,Clm9005,2024-10-23,30336.75,Approved,No
CUST1006,38,Male,Free State,49905.01,2022-12-19,POL5006,Motor,433.09,Active,Clm9006,2025-01-14,500.0,Approved,No
CUST1007,30,Female,KwaZulu-Natal,20975.4,2021-10-14,POL5007,Life,817.56,Active,Clm9007,2024-06-16,39102.08,Rejected,No
CUST1008,42,Male,North West,31654.77,2022-03-28,POL5008,Home,1410.71,Active,Clm9008,2024-02-02,48634.23,Approved,No
CUST1009,41,Male,Limpopo,26957.76,2020-11-02,POL5009,Funeral,326.92,Active,Clm9009,2024-08-05,22870.665962962965,Approved,No
